# 07 Separator (IA-SEP-001)

**Requirement:** `IA-SEP-001`

Patents: US12308414B2, EP4602674A1. Equations are paraphrased; see claims/paragraphs cited below.

$$
\newcommand{\per}{/}
\newcommand{\squared}{^{2}}
\newcommand{\cubed}{^{3}}
\newcommand{\of}[1]{\,\mathrm{#1}}
\newcommand{\milli}{\mathrm{m}}
\newcommand{\centi}{\mathrm{c}}
\newcommand{\micro}{\mathrm{\mu}}
\newcommand{\kilo}{\mathrm{k}}
\newcommand{\metre}{\mathrm{m}}
\newcommand{\gram}{\mathrm{g}}
\newcommand{\kilogram}{\mathrm{kg}}
\newcommand{\second}{\mathrm{s}}
\newcommand{\minute}{\mathrm{min}}
\newcommand{\hour}{\mathrm{h}}
\newcommand{\day}{\mathrm{d}}
\newcommand{\ampere}{\mathrm{A}}
\newcommand{\volt}{\mathrm{V}}
\newcommand{\watt}{\mathrm{W}}
\newcommand{\ohm}{\mathrm{\Omega}}
\newcommand{\kelvin}{\mathrm{K}}
\newcommand{\celsius}{{}^{\circ}\mathrm{C}}
\newcommand{\coulomb}{\mathrm{C}}
\newcommand{\joule}{\mathrm{J}}
\newcommand{\pascal}{\mathrm{Pa}}
\newcommand{\litre}{\mathrm{L}}
\newcommand{\mole}{\mathrm{mol}}
\newcommand{\molar}{\mathrm{M}}
\newcommand{\percent}{\%}
\newcommand{\atm}{\mathrm{atm}}
\newcommand{\newton}{\mathrm{N}}
\newcommand{\qty}[2]{#1\,#2}
\newcommand{\qtyrange}[3]{#1\text{--}#2\,#3}
\newcommand{\unit}[1]{#1}
\newcommand{\si}[1]{#1}
$$


## Governing equations

$$R_\mathrm{ionic} = \frac{L\,\tau}{\sigma A \varepsilon s^{1.5}}$$

Separator blocks dissolved $\mathrm{O_2}$ and bubbles from iron without a large
ionic penalty, remaining hydrophilic/macroporous (US FIG. 4A; EP Claim 2, [0048]).
Wrapped stacked-core pores $\qty{1}{\micro\metre}$ to $\qty{1}{\centi\metre}$ (US FIG. 36A).


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path

setup_path()
from plant_sim.params import CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G, default_params
from plant_sim.siunitx_nb import SIUNITX_PREAMBLE

In [ ]:
from IPython import get_ipython
from IPython.display import Markdown, display

_ipy = get_ipython()
if _ipy is not None:
    _ipy.run_line_magic("matplotlib", "inline")

sns.set(
    rc={
        "axes.labelsize": 12,
        "axes.titlesize": 18,
        "figure.figsize": (11, 8.5),
        "figure.dpi": 300,
        "figure.facecolor": "w",
        "figure.edgecolor": "k",
    }
)
display(Markdown(SIUNITX_PREAMBLE))
P = default_params()
print("KOH default", P.c_KOH_mol_m3 / 1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G, 2), round(CAPACITY_MAGNETITE_MAH_G, 2))

## Isolated component simulation

In [ ]:
from plant_sim.components.separator import Separator
from plant_sim.components.base import integrate_component

sep = Separator(P)
t, y = integrate_component(
    lambda t, y: sep.rhs(
        t,
        y,
        {"T_ely_K": P.T_ep_sim_K, "c_KOH_mol_m3": 6000, "I_cell_A": 5.0, "s_wet": 0.95, "c_O2_ely_mol_m3": 0.02},
        {},
    ),
    sep.y0(),
    (0, 800),
    n_eval=100,
)
R = [
    sep.outputs(float(ti), y[:, i], {"c_KOH_mol_m3": 6000, "I_cell_A": 5.0}, {})["R_ionic_ohm"]
    for i, ti in enumerate(t)
]
plot_frame = pd.DataFrame(
    {
        "t": (t),
        "T": (y[0]),
        "s": (y[1]),
        "c_O2": (y[2]),
        "R": (R),
    }
)
plot_long = plot_frame.melt(id_vars="t", var_name="quantity", value_name="value")
fig, ax = plt.subplots()
sns.lineplot(data=plot_long, x="t", y="value", hue="quantity", ax=ax, linewidth=2.0)
ax.set_xlabel("t (s)")
ax.set_ylabel("mixed")
ax.set_title("Separator R_ionic and O2")
ax.text(0.0, -0.22, "IA-SEP-001", transform=ax.transAxes, fontsize=8, va="top")
fig.tight_layout()
fig.subplots_adjust(bottom=0.22)

## Verification against patents / SSS002

In [ ]:
print("R_ionic is algebraic in conductivity, thickness, area, saturation.")